# Kural-TTS · Train the Sahithi voice (Kural-VITS)

The notebook covers: raw recordings on Drive → dataset → **train (auto-resume)** → ONNX export → listen.

* **Runtime:** GPU. **L4 or A100 (Colab Pro)** is recommended. A T4 works but is about 3× slower.
* **Disconnects are fine.** Checkpoints are saved to Drive every 2,000 steps, and re-running the
  *Train* cell continues from the latest one.
* Expected: about 150k–300k steps for a good voice from 15 h of data, which is several sessions.
  Listen to the eval audio in TensorBoard as you go.

## 0 · Settings

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/kural-tts"     # everything persistent lives here
RAW_DIR    = f"{DRIVE_ROOT}/recordings"             # session_01/, session_02/ ... (*.wav)
PROMPTS    = f"{DRIVE_ROOT}/prompts.tsv"            # from recording/make_script.py
DATA_DIR   = f"{DRIVE_ROOT}/data/sahithi_v1"        # built dataset (written once)
RUN_NAME   = "sahithi_v1"
RUN_DIR    = f"{DRIVE_ROOT}/runs/{RUN_NAME}"        # checkpoints + TensorBoard logs
EXPORT_DIR = f"{DRIVE_ROOT}/exports/{RUN_NAME}"
REPO_URL   = "https://github.com/Mamlesh18/kural-tts.git"
BATCH_SIZE = None     # None = config default (16). T4: 8, A100-40GB: 32

## 1 · Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)

In [ ]:
%cd /content
!test -d kural-tts && (cd kural-tts && git pull -q) || git clone -q {REPO_URL}
%cd /content/kural-tts
# Reuse Colab's CUDA PyTorch; add the package (editable) and training deps.
!pip -q install -e . librosa pyloudnorm pyyaml tensorboard onnx
!python -m pytest -q tests/training/test_vits.py   # sanity check of the model code (CPU, ~10 s)

## 2 · Build the dataset (first time, or after adding sessions)

In [ ]:
if not os.path.exists(f"{DATA_DIR}/manifests/train.jsonl"):
    !python -m training.data_pipeline.build --raw "{RAW_DIR}" --prompts "{PROMPTS}" --out "{DATA_DIR}"
else:
    print("dataset exists, delete", DATA_DIR, "to rebuild")
!head -n 30 "{DATA_DIR}/qc_report.md"

Copy the dataset to the VM's local disk, because reading thousands of small files from Drive is slow.

In [ ]:
LOCAL_DATA = "/content/data"
!rm -rf {LOCAL_DATA} && mkdir -p {LOCAL_DATA} && cp -r "{DATA_DIR}/wavs" "{DATA_DIR}/manifests" {LOCAL_DATA}/
!ls {LOCAL_DATA}/wavs | wc -l

## 3 · TensorBoard (losses + eval audio for TA / EN / code-switched / Tanglish sentences)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/logs"

## 4 · Train (re-run this cell after any disconnect; it resumes automatically)

In [ ]:
extra = f"--batch-size {BATCH_SIZE}" if BATCH_SIZE else ""
!python -m training.train --config training/configs/base.yaml --run-dir "{RUN_DIR}"     --train-manifest {LOCAL_DATA}/manifests/train.jsonl     --val-manifest {LOCAL_DATA}/manifests/val.jsonl     --audio-root {LOCAL_DATA} {extra}

## 5 · Export to ONNX (what `pip install kural-tts` runs)

In [ ]:
!python -m training.export_onnx "{RUN_DIR}" "{EXPORT_DIR}"
!ls -la "{EXPORT_DIR}"

## 6 · Listen

In [ ]:
from IPython.display import Audio, display
from kural_tts import KuralTTS

tts = KuralTTS(model_path=EXPORT_DIR)
print("providers:", tts.providers)
for text in [
    "வணக்கம், நான் சஹிதி பேசுறேன். உங்களுக்கு எப்படி உதவலாம்?",
    "Hello, this is Sahithi. Your appointment is on 15th August at 10:30 AM.",
    "நாளைக்கு meeting 10 மணிக்கு இருக்கு, please join on time.",
    "konjam wait pannunga, naan check panni solren.",
    "உங்க total amount ₹2,499, UPI மூலமா pay பண்ணலாம்.",
]:
    audio, sr = tts.synthesize(text)
    print(text)
    display(Audio(audio, rate=sr))

## 7 · Evaluate on the frozen eval set

Renders every eval sentence with the exported voice, then reuses the Phase 1 scoring (ASR CER/WER,
UTMOS) so the numbers compare directly with the baselines.

In [ ]:
!python scripts/eval/render_eval_set.py --model "{EXPORT_DIR}" --name kural_{RUN_NAME} --device cuda
!pip -q install -r scripts/baseline/envs/scoring.txt
!python scripts/baseline/score.py --utmos && cat reports/feasibility_scores.md